# Karaokê: processamento no Colab (GPU)

Este notebook baixa as músicas do YouTube, separa a voz do instrumental com IA (GPU), busca a letra
sincronizada e gera um **pacote .zip** para importar na aplicação de karaokê que roda no seu Mac.

**Antes de rodar:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.

Como importar no Mac depois:
1. Copie o `.zip` para a pasta `data/import/` do projeto.
2. No celular do anfitrião, aba com o seu nome → **Importar pacotes**. Ou, com o servidor parado:
   `uv run python -m karaoke.importer data/import/karaoke-pack-XXXX.zip`.


In [ ]:
#@title 1. Instalação (≈3 min)
# yt-dlp[default] traz o resolvedor de desafios do YouTube (yt-dlp-ejs); bgutil gera os "PO tokens"
# que o YouTube exige dos formatos quando se usa cookies ou IPs de datacenter como os do Colab.
!pip -q install -U "yt-dlp[default]" "audio-separator[gpu]" httpx bgutil-ytdlp-pot-provider
# Deno: runtime JavaScript usado pelo yt-dlp e pelo gerador de PO token
!curl -fsSL https://deno.land/install.sh | DENO_INSTALL=/usr/local sh -s -- -y >/dev/null 2>&1; deno --version | head -1
# Gerador de PO token em modo script (sem servidor em segundo plano)
![ -d /content/bgutil-ytdlp-pot-provider ] || git clone -q --depth 1 https://github.com/Brainicism/bgutil-ytdlp-pot-provider.git /content/bgutil-ytdlp-pot-provider
!cd /content/bgutil-ytdlp-pot-provider/server && deno install --allow-scripts=npm:canvas --frozen >/dev/null 2>&1 && echo "gerador de PO token pronto"
!nvidia-smi -L || echo "⚠️ Sem GPU: ative em Ambiente de execução → Alterar tipo → GPU"
!ffmpeg -version | head -1


In [ ]:
#@title 2. Músicas e opções
# Uma música por linha: link do YouTube, ID do vídeo ou texto de busca. Linhas com # são ignoradas.
# Deixe vazio para usar os pedidos feitos pelo app (MyDrive/karaoke/pedidos.txt, baixado na Biblioteca do celular).
# Prefixo "karaoke:" marca vídeos que JÁ são versão karaokê (instrumental com letra na tela):
# o notebook não separa a voz nem busca letra, e o app toca o vídeo como está.
SONGS = """
https://www.youtube.com/watch?v=qHJ8CIM4jCw
karaoke: Sing King Evidências Chitãozinho e Xororó karaoke
Legião Urbana Pais e Filhos
"""

# BS-Roformer-Viperx-1297 é o modelo com melhor qualidade de instrumental (SDR 16.5) na lista do audio-separator.
MODEL = "model_bs_roformer_ep_317_sdr_12.9755.ckpt"  #@param ["model_bs_roformer_ep_317_sdr_12.9755.ckpt", "model_bs_roformer_ep_368_sdr_12.9628.ckpt", "melband_roformer_instvox_duality_v2.ckpt", "mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt", "UVR-MDX-NET-Inst_HQ_4.onnx"]
MAX_HEIGHT = 1080  #@param [720, 1080]
# Pula vídeos que já estão no catálogo (MyDrive/karaoke/catalogo.json). Desligue para reprocessar a lista
# inteira, ou remova só alguns vídeos do catálogo na célula 5.
SKIP_DONE = True  #@param {type:"boolean"}

# Google Drive: montado já nesta célula (a autorização aparece agora, não no fim do processamento).
# Os cookies ficam guardados em MyDrive/<DRIVE_FOLDER>/cookies.txt e o pacote .zip é salvo na mesma pasta.
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "karaoke"  #@param {type:"string"}
# Se o YouTube bloquear o download no Colab ("Sign in to confirm you're not a bot" ou HTTP 403),
# exporte os cookies do YouTube numa janela anônima (extensão "Get cookies.txt LOCALLY") e feche a janela.
# Envie o arquivo uma vez para o painel de arquivos à esquerda (/content/cookies.txt): com o Drive ligado,
# ele é copiado para o Drive e nas próximas vezes é lido de lá. Enviar um arquivo novo substitui o do Drive.
COOKIES_FILE = "/content/cookies.txt"  #@param {type:"string"}
# Clientes do YouTube que o yt-dlp usa quando há cookies. "mweb" e "tv" funcionam com conta logada
# junto com o gerador de PO token; o cliente "web" com cookies só entrega formatos sem URL.
PLAYER_CLIENTS = "mweb,tv"  #@param {type:"string"}

# Velocidade: os próximos vídeos baixam em paralelo enquanto a GPU separa a música atual.
# Mais de 3 downloads simultâneos aumenta a chance de o YouTube bloquear o IP do Colab.
DOWNLOAD_WORKERS = 3  #@param {type:"slider", min:1, max:4, step:1}
# Precisão mista (FP16) na separação: costuma ser bem mais rápida na T4. Se ligar, ouça o resultado (chiado ou silêncio = desligue).
USE_AUTOCAST = False  #@param {type:"boolean"}
# Overlap da separação Roformer: 2 faz metade do trabalho na GPU do padrão do modelo (4), com perda pequena
# de qualidade nas transições. 0 = usar o padrão do modelo. Não vale para o UVR-MDX-NET.
OVERLAP = 0  #@param {type:"integer"}

# ---------- Drive e cookies ----------
import shutil
from pathlib import Path
DRIVE_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER
drive_ok = False
if USE_DRIVE:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        drive_ok = True
        print(f"Drive montado: MyDrive/{DRIVE_FOLDER}")
    except Exception as e:
        print(f"⚠️ Não deu para montar o Drive ({e}). Rode esta célula de novo e autorize com a mesma conta do"
              " Colab; sem o Drive, os cookies vêm só do Colab e o pacote é baixado pelo navegador.")

drive_cookies = DRIVE_DIR / "cookies.txt"
uploaded = Path(COOKIES_FILE) if COOKIES_FILE else None
if uploaded and uploaded.is_file():
    cookies_src = uploaded
    if drive_ok and uploaded.resolve() != drive_cookies.resolve():
        shutil.copy(uploaded, drive_cookies)
        print(f"cookies guardados no Drive: MyDrive/{DRIVE_FOLDER}/cookies.txt")
elif drive_ok and drive_cookies.is_file():
    cookies_src = drive_cookies
else:
    cookies_src = None
# o yt-dlp regrava o arquivo de cookies ao terminar (e os downloads rodam em paralelo):
# ele trabalha numa cópia local, e a cópia do Drive nunca é alterada por ele
COOKIES = ""
if cookies_src:
    COOKIES = "/content/yt-cookies.txt"
    shutil.copy(cookies_src, COOKIES)
    print(f"usando cookies de {cookies_src}")
else:
    print("sem cookies (normalmente funciona; se o YouTube bloquear, veja COOKIES_FILE acima)")

# ---------- catálogo: tudo o que já foi processado (um registro por vídeo) ----------
import json, zipfile, datetime
CATALOG_PATH = (DRIVE_DIR if drive_ok else Path("/content")) / "catalogo.json"

def catalog_entry(meta, pack, processed_at, model=None):
    v = meta.get("video") or {}
    return {"title": v.get("title") or "", "channel": v.get("channel") or "", "duration": v.get("duration"),
            "karaoke": meta.get("lyrics_source") == "video", "lyrics": bool(meta.get("lyrics")),
            "model": model, "pack": pack, "processed_at": processed_at}

def backfill_catalog(cat):
    """Lê o song.json de dentro dos pacotes que já estão no Drive (os anteriores ao catálogo)."""
    for zp in sorted(DRIVE_DIR.glob("karaoke-pack-*.zip"), reverse=True):  # o pacote mais novo vence
        try:
            with zipfile.ZipFile(zp) as z:
                try:  # a data vem do nome (karaoke-pack-AAAAMMDD-HHMM.zip); no Drive, a do arquivo é a do upload
                    when = datetime.datetime.strptime(zp.stem[-13:], "%Y%m%d-%H%M").isoformat(timespec="seconds")
                except ValueError:
                    when = datetime.datetime.fromtimestamp(zp.stat().st_mtime).isoformat(timespec="seconds")
                names = set(z.namelist())
                for n in names:
                    parts = n.split("/")
                    # só vídeos completos (sem original.mp4 o app ignora o vídeo; ele precisa ser reprocessado)
                    if len(parts) == 3 and parts[0] == "songs" and parts[2] == "song.json" and f"songs/{parts[1]}/original.mp4" in names:
                        cat["songs"].setdefault(parts[1], catalog_entry(json.loads(z.read(n)), zp.name, when))
        except (zipfile.BadZipFile, OSError, ValueError) as e:
            print(f"⚠️ não deu para ler {zp.name}: {e}")

def save_catalog():
    catalog["updated_at"] = datetime.datetime.now().isoformat(timespec="seconds")
    tmp = CATALOG_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(catalog, ensure_ascii=False, indent=1))
    tmp.replace(CATALOG_PATH)

if CATALOG_PATH.exists():
    catalog = json.loads(CATALOG_PATH.read_text())
else:
    catalog = {"updated_at": None, "songs": {}}
    if drive_ok:
        backfill_catalog(catalog)
        if catalog["songs"]:
            save_catalog()
            print(f"catálogo criado a partir dos pacotes do Drive")
print(f"catálogo: {len(catalog['songs'])} vídeo(s) já processado(s)" + ("" if drive_ok else " (só nesta sessão, sem Drive)"))

# ---------- pedidos feitos pelo app ----------
REQUESTS_PATH = DRIVE_DIR / "pedidos.txt"
if not SONGS.strip() and drive_ok and REQUESTS_PATH.exists():
    SONGS = REQUESTS_PATH.read_text()
    print(f"usando os pedidos de MyDrive/{DRIVE_FOLDER}/pedidos.txt:")
    print("\n".join("  " + l for l in SONGS.strip().splitlines() if l.strip() and not l.lstrip().startswith("#")))


In [ ]:
#@title 3. Processamento
# Pipeline: downloads e buscas de letra rodam em paralelo (DOWNLOAD_WORKERS), a GPU separa uma música
# por vez na thread principal e a codificação final roda numa thread à parte. Assim o download da
# próxima música fica escondido atrás da separação da atual.
import html, json, re, time, unicodedata, subprocess, logging, inspect, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import httpx, yt_dlp
from audio_separator.separator import Separator

WORK = Path("/content/karaoke-work"); WORK.mkdir(exist_ok=True)
MODELS = Path("/content/models"); MODELS.mkdir(exist_ok=True)

# ---------- letras (lrclib.net) ----------
_NOISE = re.compile(r"[\(\[\{][^\)\]\}]*(official|oficial|video|vídeo|clipe|lyric|letra|audio|áudio|hd|4k|remaster|ao vivo|live|visualizer|legendado|karaok[eê]|instrumental|playback|cover)[^\)\]\}]*[\)\]\}]", re.I)
_FEAT = re.compile(r"\s*(feat\.?|ft\.?|part\.?|com)\s+.*$", re.I)
_SEP = re.compile(r"\s+[-–—|]\s+")
_LRC = re.compile(r"\[(\d+):(\d+(?:\.\d+)?)\]")

# "Música - Artista - Karaokê", "Nome (Playback)", "Nome | Instrumental": tira o rótulo do fim
_TAIL = re.compile(r"\s*(?:[-–—|:]\s*)?[\(\[]?\s*(?:vers[aã]o\s+)?(?:karaok[eê]|playback|instrumental|com letra|cover)\s*[\)\]]?\s*$", re.I)

def clean_title(t):
    t = _NOISE.sub("", unicodedata.normalize("NFC", html.unescape(t)))
    for _ in range(3):
        t2 = _TAIL.sub("", t)
        if t2 == t: break
        t = t2
    t = re.sub(r"\s*[\(\[\{]\s*[\)\]\}]", "", t); t = re.sub(r'["“”]', "", t)
    return re.sub(r"\s{2,}", " ", t).strip(" -–|:")

def title_queries(title):
    """Consultas candidatas: 'Artista Música' e 'Música Artista' (a ordem no título varia) e o título limpo."""
    c = clean_title(title)
    parts = [x.strip() for x in _SEP.split(c) if x.strip()]
    queries = []
    if len(parts) >= 2:
        a, b = parts[0], _FEAT.sub("", " ".join(parts[1:])).strip()
        queries += [f"{a} {b}", f"{b} {a}"]
    if c and c not in queries: queries.append(c)
    return queries

def parse_lrc(text):
    lines = []
    for raw in text.splitlines():
        stamps = _LRC.findall(raw)
        if not stamps: continue
        content = _LRC.sub("", raw).strip()
        for mm, ss in stamps: lines.append({"t": int(mm) * 60 + float(ss), "text": content, "singer": None})
    return sorted(lines, key=lambda l: l["t"])

_lrclib_lock = threading.Lock()  # uma busca por vez: o lrclib limita as requisições (HTTP 429)

def lrclib_search(client, q):
    """Busca no lrclib esperando 2, 4, 8, 16 s quando ele limita ou falha; None se não houve resposta."""
    for attempt in range(5):
        try:
            with _lrclib_lock:
                r = client.get("https://lrclib.net/api/search", params={"q": q})
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(min(2.0 * 2**attempt, 30)); continue
            items = r.json()
            return items if isinstance(items, list) else None
        except Exception:
            time.sleep(2.0 * 2**attempt)
    return None

def fetch_lyrics(title, duration):
    unanswered = 0
    with httpx.Client(timeout=15, headers={"User-Agent": "karaoke-local/0.1 colab"}) as client:
        for q in title_queries(title):
            items = lrclib_search(client, q)
            if items is None:
                unanswered += 1
                continue
            best = None
            for it in items:
                if not it.get("syncedLyrics") or it.get("instrumental"): continue
                score = abs(float(it["duration"]) - duration) if duration and it.get("duration") else 4.0
                if score > 8: continue
                if best is None or score < best[0]: best = (score, it)
            if best:
                lines = parse_lrc(best[1]["syncedLyrics"])
                if lines: return lines, f"lrclib: {best[1].get('artistName')} - {best[1].get('trackName')}"
    # "none" = o lrclib não tem a letra; sem resposta, o app no Mac tenta de novo quando a música entra na fila
    return None, ("erro: lrclib sem resposta" if unanswered else "none")

# ---------- youtube ----------
POT_SERVER = "/content/bgutil-ytdlp-pot-provider/server"

class _YtLogger:
    """Guarda os avisos do yt-dlp de uma música para mostrar só quando o download dela falha."""
    def __init__(self, warnings): self.warnings = warnings
    def debug(self, msg):
        if msg.startswith("[debug]"): return
        self.info(msg)
    def info(self, msg): pass
    def warning(self, msg): self.warnings.append(msg)
    def error(self, msg): self.warnings.append(msg)

def yt_opts(warnings, **extra):
    opts = {"quiet": True, "noplaylist": True, "logger": _YtLogger(warnings),
            "js_runtimes": {"deno": {}, "node": {}},
            "extractor_args": {"youtubepot-bgutilscript": {"server_home": [POT_SERVER]}}}
    if COOKIES:
        opts["cookiefile"] = COOKIES
        clients = [c.strip() for c in PLAYER_CLIENTS.split(",") if c.strip()]
        if clients: opts["extractor_args"]["youtube"] = {"player_client": clients}
    opts.update(extra)
    return opts

def resolve(query, warnings):
    opts = yt_opts(warnings, skip_download=True, extract_flat="in_playlist")
    m = re.search(r"(?:v=|youtu\.be/|shorts/|embed/)([A-Za-z0-9_-]{11})", query)
    target = f"https://www.youtube.com/watch?v={m.group(1)}" if m else (f"https://www.youtube.com/watch?v={query}" if re.fullmatch(r"[A-Za-z0-9_-]{11}", query) else f"ytsearch1:{query}")
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(target, download=False)
    if "entries" in info: info = info["entries"][0]
    return info["id"]

def download(yt_id, dest, warnings):
    h = MAX_HEIGHT
    opts = yt_opts(warnings, noprogress=True, overwrites=True,
            format=f"bestvideo[height<={h}][ext=mp4][vcodec^=avc1]+bestaudio[ext=m4a]/bestvideo[height<={h}][ext=mp4]+bestaudio[ext=m4a]/bestvideo[height<={h}]+bestaudio/best[height<={h}]/bestvideo*+bestaudio/best",
            merge_output_format="mp4", outtmpl=str(dest.with_suffix("")) + ".%(ext)s")
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(f"https://www.youtube.com/watch?v={yt_id}", download=True)
    return {"yt_id": yt_id, "title": unicodedata.normalize("NFC", html.unescape(info.get("title") or yt_id)),
            "channel": info.get("channel") or info.get("uploader") or "", "duration": info.get("duration"),
            "thumbnail": f"https://i.ytimg.com/vi/{yt_id}/hqdefault.jpg"}

def ffmpeg(*args):
    subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", *args], check=True)

# ---------- separação (só na thread principal: o Separator guarda a pasta de saída como estado) ----------
def separator_options():
    """Autocast (FP16) e overlap, só quando a versão instalada do audio-separator aceita."""
    params = inspect.signature(Separator).parameters
    opts = {}
    if "use_autocast" in params: opts["use_autocast"] = USE_AUTOCAST
    elif USE_AUTOCAST: print("aviso: esta versão do audio-separator não tem use_autocast")
    # overlap dos Roformer (mdxc_params): quantas janelas cobrem cada trecho; o padrão vem do modelo (4 no BS-Roformer)
    default = params["mdxc_params"].default if "mdxc_params" in params else None
    if int(OVERLAP) > 0:
        if isinstance(default, dict) and "overlap" in default: opts["mdxc_params"] = {**default, "overlap": int(OVERLAP)}
        else: print("aviso: esta versão do audio-separator não aceita overlap; usando o do modelo")
    return opts

separator = None
def separate(audio, out_dir):
    global separator
    if separator is None:
        separator = Separator(log_level=logging.WARNING, model_file_dir=str(MODELS), output_dir=str(out_dir),
                              output_format="WAV", **separator_options())
        separator.load_model(model_filename=MODEL)
    separator.output_dir = str(out_dir)
    separator.model_instance.output_dir = str(out_dir)
    separator.separate(str(audio), custom_output_names={"Instrumental": "instrumental", "Vocals": "vocals"})
    return out_dir / "instrumental.wav", out_dir / "vocals.wav"

# ---------- pipeline ----------
_print_lock = threading.Lock()
_claim_lock = threading.Lock()
claimed = set()  # yt_ids já pegos por algum item da lista (dois itens podem apontar para o mesmo vídeo)

class Job:
    def __init__(self, raw):
        self.is_karaoke = raw.lower().startswith("karaoke:")
        self.query = raw.split(":", 1)[1].strip() if self.is_karaoke else raw
        self.yt_id = self.dir = self.video = self.lyrics = self.source = self.error = None
        self.skip = False         # já processado ou repetido na lista
        self.needs_separation = False  # precisa passar pela GPU
        self.warnings = []
        self.t0 = time.time()

    def say(self, msg):
        with _print_lock: print(f"[{self.yt_id or self.query[:24]}] {msg}", flush=True)

    def fail(self, stage, e):
        self.error = f"{stage}: {type(e).__name__}: {str(e)[:200]}"
        self.say(f"✗ erro na {self.error}")
        relevant = [w for w in self.warnings if re.search(r"PO Token|SABR|Sign in|cookies|JavaScript|format", w, re.I)]
        for w in relevant[-3:]: self.say(f"   aviso yt-dlp: {w[:220]}")
        if "Requested format is not available" in str(e):
            self.say("   dica: o YouTube não entregou formatos. Confira a célula 1 (deno e gerador de PO token) e,"
                     " se estiver usando cookies, exporte-os de novo numa janela anônima.")

def prepare(raw):
    """Rede e CPU: resolve, baixa, extrai o áudio e busca a letra. Roda em paralelo."""
    job = Job(raw)
    try:
        job.yt_id = resolve(job.query, job.warnings)
        with _claim_lock:
            if job.yt_id in claimed:
                job.skip = True; job.say("repetido na lista, pulando"); return job
            claimed.add(job.yt_id)
        done = catalog["songs"].get(job.yt_id)
        if SKIP_DONE and done:
            job.skip = True; job.say(f"✓ já processado ({done.get('pack') or 'catálogo'}), pulando"); return job
        job.dir = WORK / job.yt_id; job.dir.mkdir(exist_ok=True)
        if (job.dir / "song.json").exists():
            job.skip = True; job.say("✓ já processado, pulando"); return job
        job.say(f"▶ {job.query[:60]}")
        # cada etapa reaproveita o que já existe na pasta (útil ao rodar de novo depois de um erro)
        info_path = job.dir / "meta.json"
        if (job.dir / "original.mp4").exists() and info_path.exists():
            job.video = json.loads(info_path.read_text()); job.say(f"   download reaproveitado: {job.video['title']}")
        else:
            job.video = download(job.yt_id, job.dir / "original.mp4", job.warnings)
            info_path.write_text(json.dumps(job.video, ensure_ascii=False))
            job.say(f"   baixado: {job.video['title']} ({job.video['duration']}s)")
        if not job.is_karaoke:
            if (job.dir / "instrumental.mp4").exists() and (job.dir / "vocals.m4a").exists():
                job.say("   separação reaproveitada")
            else:
                ffmpeg("-i", str(job.dir / "original.mp4"), "-vn", "-ac", "2", "-ar", "44100", str(job.dir / "audio.wav"))
                job.needs_separation = True
        if job.is_karaoke:
            job.lyrics, job.source = None, "video"  # a letra já está no próprio vídeo
        else:
            try:
                job.lyrics, job.source = fetch_lyrics(job.video["title"], job.video["duration"])
            except Exception as e:  # letra é opcional: nunca derruba a música
                job.lyrics, job.source = None, f"erro ({type(e).__name__})"
        job.say(f"   letra: {job.source}")
    except Exception as e:
        job.fail("preparação", e)
    return job

def finish(job, inst=None, voc=None):
    """CPU: junta o instrumental ao vídeo, codifica a voz e grava o song.json."""
    try:
        d = job.dir
        if inst:
            ffmpeg("-i", str(d / "original.mp4"), "-i", str(inst), "-map", "0:v:0", "-map", "1:a:0", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest", str(d / "instrumental.mp4"))
            ffmpeg("-i", str(voc), "-c:a", "aac", "-b:a", "160k", str(d / "vocals.m4a"))
            for f in (inst, voc, d / "audio.wav"): f.unlink(missing_ok=True)
        song = {"video": job.video, "status": "ready", "progress": 1.0, "error": None,
                "has_original": True, "has_instrumental": not job.is_karaoke, "has_vocals": not job.is_karaoke,
                "lyrics": job.lyrics, "lyrics_source": job.source}
        (d / "song.json").write_text(json.dumps(song, ensure_ascii=False, indent=1))
        job.say(f"   ok em {time.time()-job.t0:.0f}s")
        return job.yt_id
    except Exception as e:
        job.fail("codificação", e)

jobs = [l.strip() for l in SONGS.strip().splitlines() if l.strip() and not l.strip().startswith("#")]
results = []
started = time.time()
# uma thread de codificação basta: ela leva menos que a separação, e o Colab gratuito tem só 2 vCPUs
downloads, encodes = ThreadPoolExecutor(max(1, int(DOWNLOAD_WORKERS))), ThreadPoolExecutor(1)
try:
    prepared = [downloads.submit(prepare, raw) for raw in jobs]
    finishing = []
    # na ordem em que os downloads terminam: um vídeo lento não deixa a GPU parada esperando por ele
    for fut in as_completed(prepared):
        job = fut.result()
        if job.error: continue
        if job.skip:
            if (job.dir is not None) and (job.dir / "song.json").exists(): finishing.append(job.yt_id)
            continue
        inst = voc = None
        if job.needs_separation:
            try:
                job.say("   separando a voz na GPU…")
                inst, voc = separate(job.dir / "audio.wav", job.dir)
                job.say("   voz separada")
            except Exception as e:
                job.fail("separação", e); continue
        finishing.append(encodes.submit(finish, job, inst, voc))
    for item in finishing:
        yt_id = item if isinstance(item, str) else item.result()
        if yt_id and yt_id not in results: results.append(yt_id)
except KeyboardInterrupt:  # botão de parar do Colab: descarta os downloads que ainda nem começaram
    print("interrompido; os arquivos já baixados ficam na pasta e são reaproveitados na próxima execução")
    raise
finally:
    downloads.shutdown(wait=False, cancel_futures=True); encodes.shutdown(wait=False, cancel_futures=True)

print(f"\n{len(results)} música(s) prontas em {time.time()-started:.0f}s: {results}")


In [ ]:
#@title 4. Gerar o pacote .zip
import zipfile, datetime, re
from pathlib import Path
now = datetime.datetime.now()
name = f"karaoke-pack-{now:%Y%m%d-%H%M}.zip"
out = Path("/content") / name
# o catálogo vai atualizado dentro do pacote: o app no Mac fica sabendo de tudo o que já foi processado
for yt_id in results:
    meta = json.loads((WORK / yt_id / "song.json").read_text())
    karaoke = meta.get("lyrics_source") == "video"
    catalog["songs"][yt_id] = catalog_entry(meta, name, now.isoformat(timespec="seconds"), None if karaoke else MODEL)
catalog["updated_at"] = now.isoformat(timespec="seconds")
with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_STORED) as z:
    for yt_id in results:
        d = WORK / yt_id
        for f in ("song.json", "original.mp4", "instrumental.mp4", "vocals.m4a"):
            if (d / f).exists():
                z.write(d / f, f"songs/{yt_id}/{f}")
    z.writestr("catalogo.json", json.dumps(catalog, ensure_ascii=False, indent=1))
print(f"{out} ({out.stat().st_size/1e6:.0f} MB)")
save_catalog()
print(f"catálogo: {len(catalog['songs'])} vídeo(s)")

# tira dos pedidos o que já está no catálogo (os blocos são "# título" + link)
if drive_ok and REQUESTS_PATH.exists():
    keep, block = [], []
    for line in REQUESTS_PATH.read_text().splitlines():
        block.append(line)
        if line.strip() and not line.lstrip().startswith("#"):
            m = re.search(r"(?:v=|youtu\.be/|shorts/)([A-Za-z0-9_-]{11})", line)
            if not (m and m.group(1) in catalog["songs"]): keep += block
            block = []
    left = sum(1 for l in keep if l.strip() and not l.lstrip().startswith("#"))
    if left: REQUESTS_PATH.write_text("\n".join(keep) + "\n")
    else: REQUESTS_PATH.unlink()
    print(f"pedidos restantes: {left}")

saved = False
if drive_ok:  # montado na célula 2
    try:
        import shutil; shutil.copy(out, DRIVE_DIR / name)
        print(f"copiado para o Drive: MyDrive/{DRIVE_FOLDER}/{name}")
        saved = True
    except OSError as e:
        print(f"⚠️ falha ao copiar para o Drive ({e}); baixando pelo navegador")
if not saved:
    from google.colab import files
    files.download(str(out))


In [ ]:
#@title 5. Catálogo (consultar e remover)
# Lista o que já foi processado. Para reprocessar um vídeo, informe o ID (ou o link) em REMOVE e rode:
# ele sai do catálogo e deixa de ser pulado na célula 3. O pacote antigo no Drive não é apagado.
FILTER = ""  #@param {type:"string"}
REMOVE = ""  #@param {type:"string"}
import re
for raw in re.split(r"[\s,]+", REMOVE.strip()):
    if not raw: continue
    m = re.search(r"(?:v=|youtu\.be/|shorts/)([A-Za-z0-9_-]{11})", raw)
    yt_id = m.group(1) if m else raw
    if catalog["songs"].pop(yt_id, None): print(f"removido do catálogo: {yt_id}")
    else: print(f"não está no catálogo: {yt_id}")
if REMOVE.strip(): save_catalog()

rows = sorted(catalog["songs"].items(), key=lambda kv: kv[1].get("processed_at") or "", reverse=True)
f = FILTER.strip().lower()
rows = [(k, v) for k, v in rows if not f or f in f"{v['title']} {v['channel']} {k}".lower()]
print(f"{len(rows)} de {len(catalog['songs'])} vídeo(s)\n")
for yt_id, v in rows:
    tags = ["karaokê" if v.get("karaoke") else "separada", "letra" if v.get("lyrics") else "sem letra"]
    print(f"{yt_id}  {(v.get('processed_at') or '')[:10]}  {v['title'][:60]}")
    print(f"             {v.get('channel') or ''} · {' · '.join(tags)} · {v.get('pack') or ''}")


In [ ]:
#@title 6. Analisar os pacotes do Drive
# Independente das outras células: monta o Drive, lê só o índice e o song.json de cada .zip (rápido mesmo
# com pacotes de GB) e mostra o que tem em cada um, os vídeos repetidos, os problemas e a diferença para o
# catálogo. Com UPDATE_CATALOG, os vídeos que estão nos pacotes e faltam no catálogo são adicionados.
DRIVE_FOLDER = "karaoke"  #@param {type:"string"}
UPDATE_CATALOG = True  #@param {type:"boolean"}
SHOW_SONGS = True  #@param {type:"boolean"}

import json, zipfile, datetime
from pathlib import Path
from collections import defaultdict
from google.colab import drive
drive.mount("/content/drive")
pack_dir = Path("/content/drive/MyDrive") / DRIVE_FOLDER
cat_path = pack_dir / "catalogo.json"
cat = json.loads(cat_path.read_text()) if cat_path.exists() else {"updated_at": None, "songs": {}}

def pack_date(zp):
    try: return datetime.datetime.strptime(zp.stem[-13:], "%Y%m%d-%H%M").isoformat(timespec="seconds")
    except ValueError: return datetime.datetime.fromtimestamp(zp.stat().st_mtime).isoformat(timespec="seconds")

MB = 1e6
found = defaultdict(list)   # yt_id -> [(pack, song.json, data, válido)] do mais novo para o mais antigo
problems = []
packs = sorted(pack_dir.glob("karaoke-pack-*.zip"), reverse=True)
print(f"MyDrive/{DRIVE_FOLDER}: {len(packs)} pacote(s), catálogo com {len(cat['songs'])} vídeo(s)\n")
for zp in packs:
    try:
        with zipfile.ZipFile(zp) as z:
            files = defaultdict(dict)
            for info in z.infolist():
                parts = info.filename.split("/")
                if len(parts) == 3 and parts[0] == "songs" and parts[2]:
                    files[parts[1]][parts[2]] = info.file_size
            has_cat = "catalogo.json" in z.namelist()
            print(f"📦 {zp.name}  {zp.stat().st_size/MB:,.0f} MB  {len(files)} vídeo(s)" + ("  (com catálogo)" if has_cat else ""))
            for yt_id, fs in sorted(files.items()):
                meta = {}
                if "song.json" in fs:
                    try: meta = json.loads(z.read(f"songs/{yt_id}/song.json"))
                    except (ValueError, KeyError) as e: problems.append(f"{zp.name}: song.json inválido em {yt_id} ({e})")
                else:
                    problems.append(f"{zp.name}: {yt_id} sem song.json (o app ignora)")
                if "original.mp4" not in fs:
                    problems.append(f"{zp.name}: {yt_id} sem original.mp4 (o app ignora)")
                v = meta.get("video") or {}
                karaoke = meta.get("lyrics_source") == "video"
                if not karaoke and meta and not ("instrumental.mp4" in fs and "vocals.m4a" in fs):
                    problems.append(f"{zp.name}: {yt_id} sem instrumental/voz separada ({v.get('title', '')[:40]})")
                found[yt_id].append((zp.name, meta, pack_date(zp), bool(meta) and "original.mp4" in fs))
                if SHOW_SONGS:
                    tags = ["karaokê" if karaoke else "separada", "letra" if meta.get("lyrics") else "sem letra"]
                    size = sum(fs.values()) / MB
                    print(f"   {yt_id}  {(v.get('title') or '?')[:55]}  ·  {' · '.join(tags)} · {size:,.0f} MB")
    except (zipfile.BadZipFile, OSError) as e:
        problems.append(f"{zp.name}: não abriu ({e})")
    print()

dups = {k: [p for p, *_ in v] for k, v in found.items() if len(v) > 1}
if dups:
    print(f"🔁 {len(dups)} vídeo(s) em mais de um pacote (o mais novo primeiro):")
    for yt_id, ps in dups.items():
        title = (found[yt_id][0][1].get("video") or {}).get("title", "")
        print(f"   {yt_id}  {title[:50]}: {', '.join(ps)}")
    print()
if problems:
    print(f"⚠️ {len(problems)} problema(s):")
    for p in problems: print("   " + p)
    print()

pack_names = {zp.name for zp in packs}
valid = {k: next((x for x in v if x[3]), None) for k, v in found.items()}  # cópia mais nova que o app aceita
missing_in_cat = [k for k, x in valid.items() if x and k not in cat["songs"]]
broken = [k for k, x in valid.items() if x is None]
orphans = [k for k, e in cat["songs"].items() if e.get("pack") and e["pack"] not in pack_names]
print(f"Vídeos únicos nos pacotes: {len(found)} · no catálogo: {len(cat['songs'])}")
if broken:
    print(f"   {len(broken)} sem nenhuma cópia válida (ficam fora do catálogo para serem reprocessados): {', '.join(broken)}")
if missing_in_cat:
    print(f"   {len(missing_in_cat)} nos pacotes e fora do catálogo: {', '.join(missing_in_cat)}")
if orphans:
    print(f"   {len(orphans)} no catálogo apontando para pacote que não está mais no Drive: {', '.join(orphans)}")
if not missing_in_cat and not orphans:
    print("   catálogo e pacotes batem")

if UPDATE_CATALOG and missing_in_cat:
    for yt_id in missing_in_cat:
        pack, meta, when, _ = valid[yt_id]
        v = meta.get("video") or {}
        cat["songs"][yt_id] = {"title": v.get("title") or "", "channel": v.get("channel") or "",
                               "duration": v.get("duration"), "karaoke": meta.get("lyrics_source") == "video",
                               "lyrics": bool(meta.get("lyrics")), "model": None, "pack": pack, "processed_at": when}
    cat["updated_at"] = datetime.datetime.now().isoformat(timespec="seconds")
    tmp = cat_path.with_suffix(".tmp"); tmp.write_text(json.dumps(cat, ensure_ascii=False, indent=1)); tmp.replace(cat_path)
    if "catalog" in globals(): catalog = cat  # mantém a célula 4 em dia se a célula 2 já rodou nesta sessão
    print(f"\n✓ catálogo atualizado: {len(missing_in_cat)} vídeo(s) adicionado(s), {len(cat['songs'])} no total")


## Observações

- A primeira música demora mais (download do modelo, ~600 MB). Depois, cada música de 4 min leva
  cerca de 20 a 40 s na GPU T4.
- Os downloads rodam em paralelo com a separação (`DOWNLOAD_WORKERS`), então numa lista longa o tempo
  total fica perto do tempo só de GPU. As mensagens de músicas diferentes se intercalam, cada uma com o ID do vídeo.
- Se o download falhar com mensagem de robô ou HTTP 403, use cookies (ver `COOKIES_FILE` na célula 2). Com
  o Drive ligado, basta enviar o `cookies.txt` uma vez: ele fica em `MyDrive/karaoke/cookies.txt` e é lido de
  lá nas próximas execuções. Os cookies dão acesso à sua sessão do YouTube: não compartilhe essa pasta.
  Com cookies, o yt-dlp passa a usar os clientes `mweb` e `tv` com o gerador de PO token instalado na célula 1.
- "Requested format is not available" significa que o YouTube não entregou formato nenhum: rode a célula 1
  de novo (deno + gerador de PO token) e, se usar cookies, exporte-os de novo numa janela anônima.
- O zip não é comprimido (vídeo já é comprimido); ~50 a 150 MB por música em 1080p.
- Rode a célula 4 quantas vezes quiser: ela só inclui as músicas da lista `results` da célula 3.
- Catálogo: `MyDrive/karaoke/catalogo.json` registra cada vídeo processado e em qual pacote ele está.
  Na primeira execução ele é montado a partir dos pacotes que já estão no Drive. A célula 3 pula o que está
  nele (`SKIP_DONE`), a célula 5 lista e remove entradas, e cada pacote leva uma cópia do catálogo para o app.
- A célula 6 roda sozinha (não precisa das outras): abre cada pacote do Drive e mostra os vídeos de cada um,
  os repetidos entre pacotes, os problemas (zip corrompido, vídeo sem `original.mp4` ou sem voz separada) e a
  diferença para o catálogo. Com `UPDATE_CATALOG`, completa o catálogo com o que está nos pacotes.
- Pedidos: no celular, o anfitrião usa "Pedir ao Colab" na busca e, na Biblioteca → Pedidos, baixa o
  `pedidos.txt`. Salve-o em `MyDrive/karaoke/` e deixe `SONGS` vazio: o notebook processa os pedidos e
  tira do arquivo os que ficaram prontos.
